# Elliptic Dataset — Exploratory Data Analysis

Covers: class balance overall and per time step, whether edges ever cross time steps, degree distribution, neighbor-label homophily (does illicit cluster locally?), and feature distributions split by class. These numbers justify (or don't) using a GNN in Phase 3, and directly shaped how Phase 4 was framed (see section 3).

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data import ILLICIT, LICIT, UNKNOWN, load_raw

sns.set_theme(style='whitegrid')
data = load_raw()
print(f"nodes={len(data.features)}  edges={len(data.edges)}  features={len(data.feature_cols)}")

## 1. Class balance — overall

In [ ]:
counts = data.labels.value_counts().rename({ILLICIT: 'illicit', LICIT: 'licit', UNKNOWN: 'unknown'})
display(counts)
display((counts / counts.sum() * 100).round(2).rename('pct'))

**Findings:** 203,769 nodes: 4,545 illicit (2.23%), 42,019 licit (20.6%), 157,205 unknown (77.2%) — matches the PRD's stated ~2/21/77 split exactly.

## 2. Class balance per time step

Checks whether every time step has enough illicit examples for a usable split (informs the exact Phase 1 train/val/test cut).

In [ ]:
df = data.features[['time_step']].copy()
df['label'] = data.labels.values
per_step = df.groupby(['time_step', 'label']).size().unstack(fill_value=0)
per_step = per_step.rename(columns={ILLICIT: 'illicit', LICIT: 'licit', UNKNOWN: 'unknown'})

fig, ax = plt.subplots(figsize=(12, 4))
per_step[['illicit', 'licit']].plot(ax=ax)
ax.axvline(34.5, color='gray', linestyle='--', label='train/val cut')
ax.axvline(39.5, color='black', linestyle='--', label='val/test cut')
ax.set_ylabel('# labeled nodes')
ax.legend()
plt.show()
per_step

**Findings:** the illicit rate spikes sharply at time step 13 (36.0% of labeled nodes — the known dark-market-takedown period in this dataset), far above the ~11% average illicit rate among labeled nodes elsewhere. The test window (steps 40-49) has illicit counts ranging from a healthy 239 (step 42) down to a thin 2 (step 46) and 5 (step 45) — usable in aggregate (636 illicit test nodes total) but per-time-step metrics for those two steps individually are noisy; flagged again in Phase 6's per-time-step breakdown.

## 3. Do edges ever cross time steps?

This matters more than it looks: if every edge stays within a single time step, the transaction graph is really 49 disconnected components, one per time step — which changes what "leakage" can even mean for Phase 4's inductive protocol (see the markdown note after this cell).

In [ ]:
ts = data.features['time_step']
src_ts = data.edges['src'].map(ts)
dst_ts = data.edges['dst'].map(ts)
cross_time = (src_ts != dst_ts)
print(f"edges crossing a time-step boundary: {cross_time.sum()} / {len(data.edges)}")

**Findings: 0 of 234,355 edges cross a time-step boundary.** Every transaction edge connects two nodes within the *same* time step. This is a real, verified property of the public Elliptic release, not a bug in the loader (checked directly against the raw CSVs, bypassing `src/data.py` entirely).

**Why this matters for Phase 4:** the literature's framing of transductive "leakage" (a GCN's message passing reaching forward across a directed, time-ordered edge into a not-yet-seen future transaction) assumes edges that span time. They don't, here — so a node's k-hop neighborhood can *never* include a node from a different time step, regardless of protocol. Phase 4's causal edge filter (`time[src] <= time[dst]`) is consequently a no-op on this dataset (verified empirically: it keeps 100% of edges, symmetrized or not), and the transductive vs. inductive GNN numbers in `results/leakage_gap_analysis.md` are correspondingly close — a genuine finding, not an implementation gap. The more informative test given this property is the edge-shuffle ablation (Phase 4, task 4): whether the *within-time-step* topology carries real signal at all, independent of any temporal-leakage question.

## 4. Degree distribution

In [ ]:
deg = pd.concat([data.edges['src'], data.edges['dst']]).value_counts()
deg = deg.reindex(data.features.index, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(deg, bins=50, log=True)
ax.set_xlabel('degree')
ax.set_ylabel('count (log scale)')
ax.set_title('Node degree distribution')
plt.show()
print(deg.describe())

## 5. Neighbor-label homophily

For each labeled node, what fraction of its neighbors share its label? High homophily is the main justification for a GNN — if illicit nodes cluster together, message passing should help; if not, a GNN has little structural signal to exploit.

In [ ]:
id_to_idx = {tx_id: i for i, tx_id in enumerate(data.features.index)}
labels_arr = data.labels.to_numpy()
n = len(labels_arr)

neighbor_lists = [[] for _ in range(n)]
src_idx = data.edges['src'].map(id_to_idx).to_numpy()
dst_idx = data.edges['dst'].map(id_to_idx).to_numpy()
for s, d in zip(src_idx, dst_idx):
    neighbor_lists[s].append(d)
    neighbor_lists[d].append(s)

homophily = []
for i in range(n):
    if labels_arr[i] == UNKNOWN or not neighbor_lists[i]:
        continue
    neigh_labels = labels_arr[neighbor_lists[i]]
    known = neigh_labels[neigh_labels != UNKNOWN]
    if len(known) == 0:
        continue
    homophily.append({'label': labels_arr[i], 'frac_same_label': (known == labels_arr[i]).mean()})

homophily_df = pd.DataFrame(homophily)
homophily_df.groupby('label')['frac_same_label'].mean().rename({ILLICIT: 'illicit', LICIT: 'licit'})

**Findings:** illicit nodes have 51.0% same-label neighbors on average (n=2,484 illicit nodes with at least one known-label neighbor) vs. 98.2% for licit nodes (n=33,390). The licit number is inflated by base rate (licit is ~90% of *labeled* nodes, so even random neighbors would look highly "licit-homophilous"); the illicit number — roughly 5x the ~11% illicit base rate among labeled nodes — is the more meaningful signal: illicit transactions really do cluster with other illicit transactions well above chance, which is the actual justification for trying a GNN in Phase 3. Given section 3's finding, that clustering signal is necessarily local to a single time step.

## 6. Feature distributions by class

A handful of the most separating features (proxied here by absolute mean difference between classes).

In [ ]:
labeled = data.labeled_mask()
X = data.features.loc[labeled, data.feature_cols]
y = data.labels.loc[labeled]

mean_diff = (X[y == ILLICIT].mean() - X[y == LICIT].mean()).abs().sort_values(ascending=False)
top_features = mean_diff.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flat, top_features):
    sns.kdeplot(X.loc[y == LICIT, feat], ax=ax, label='licit')
    sns.kdeplot(X.loc[y == ILLICIT, feat], ax=ax, label='illicit')
    ax.set_title(feat)
    ax.legend()
plt.tight_layout()
plt.show()

**Findings:** cross-check the features that show the clearest licit/illicit separation here (by raw mean-difference) against `results/figures/xgb_feature_importance.png` from Phase 2 — the two rankings won't be identical (this cell doesn't account for feature scale or interaction effects the way XGBoost's gain importance does), but strong overlap corroborates that the baseline model is keying off real, visually-verifiable separation rather than noise.